In [1]:
# Insider Trading Detection — Data Collection Notebook (Enhanced)
# Includes Selenium + NSE Authenticated Fetch

# -----------------------------
# 0. Overview
# -----------------------------
# Collects datasets for an Insider Trading Detection system tailored to Indian markets:
#  - Market data (OHLCV) via yfinance
#  - SEBI insider disclosures (via Selenium)
#  - NSE corporate announcements (authenticated API)
#  - Basic relational edge-list for GraphSAGE
#
# Data saved under ./data/{market,insider,announcements,graphs}

# -----------------------------
# 1. Environment / Installs
# -----------------------------
# !pip install yfinance pandas requests beautifulsoup4 tqdm lxml selenium webdriver-manager nsetools
# !pip install transformers torch sentence-transformers

import os, time, json, requests, datetime as dt
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import yfinance as yf

# -----------------------------
# 2. Create data folders
# -----------------------------
DATA_DIR = "./data"
MARKET_DIR = os.path.join(DATA_DIR, "market")
INSIDER_DIR = os.path.join(DATA_DIR, "insider")
ANN_DIR = os.path.join(DATA_DIR, "announcements")
GRAPH_DIR = os.path.join(DATA_DIR, "graphs")
for p in [DATA_DIR, MARKET_DIR, INSIDER_DIR, ANN_DIR, GRAPH_DIR]:
    os.makedirs(p, exist_ok=True)

# -----------------------------
# 3. Fetch Market Data (yfinance)
# -----------------------------
def fetch_and_store_market_data(tickers, start="2018-01-01", end=None, interval="1d"):
    if end is None:
        end = dt.date.today().isoformat()
    all_meta = {}
    for t in tqdm(tickers, desc="Downloading tickers"):
        try:
            df = yf.download(t, start=start, end=end, interval=interval, progress=False, threads=False)
            if df is None or df.shape[0] == 0:
                print(f"No data for {t}")
                continue
            fname = os.path.join(MARKET_DIR, f"{t.replace('.','_')}_{interval}.csv")
            df.to_csv(fname)
            all_meta[t] = {"rows": len(df), "path": fname}
            time.sleep(0.5)
        except Exception as e:
            print(f"Failed {t}: {e}")
    pd.DataFrame.from_dict(all_meta, orient="index").to_csv(os.path.join(MARKET_DIR, "meta.csv"))
    return all_meta

# -----------------------------
# 4. Fetch SEBI Insider Disclosures (Selenium)
# -----------------------------
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

def fetch_sebi_disclosures_selenium(save_csv=True):
    options = Options()
    options.add_argument('--headless')
    driver = webdriver.Chrome(options=options)
    driver.get('https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doPmr=yes')
    time.sleep(5)

    soup = BeautifulSoup(driver.page_source, 'lxml')
    driver.quit()

    rows = []
    for table in soup.find_all('table'):
        for tr in table.find_all('tr'):
            cells = [c.get_text(strip=True) for c in tr.find_all(['td','th'])]
            if cells:
                rows.append(cells)

    if not rows:
        print('⚠️ No SEBI table rows found — check site structure.')
        return None

    df = pd.DataFrame(rows)
    if save_csv:
        df.to_csv(os.path.join(INSIDER_DIR, 'sebi_disclosures.csv'), index=False)
        print(f'✅ Saved SEBI disclosures: {df.shape[0]} rows.')
    return df

# -----------------------------
# 5. Fetch NSE Corporate Announcements (Authenticated)
# -----------------------------
def fetch_nse_announcements(save_csv=True):
    url = 'https://www.nseindia.com/api/corporate-announcements?index=equities'
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Accept': 'application/json, text/plain, */*',
        'Referer': 'https://www.nseindia.com/'
    }
    s = requests.Session()
    s.get('https://www.nseindia.com', headers=headers)
    time.sleep(1)
    r = s.get(url, headers=headers)
    if r.status_code != 200:
        print('⚠️ NSE API returned', r.status_code)
        return None
    data = r.json()
    df = pd.json_normalize(data['data']) if 'data' in data else pd.DataFrame(data)
    if save_csv:
        df.to_csv(os.path.join(ANN_DIR, 'nse_announcements.csv'), index=False)
        print(f'✅ Saved {len(df)} NSE announcements.')
    return df

# -----------------------------
# 6. Build Graph Edge List (Insider -> Company)
# -----------------------------
def build_graph_from_sebi(disclosure_df, save=True):
    df = disclosure_df.copy()
    possible_insider = [c for c in df.columns if 'insider' in c.lower() or 'name' in c.lower() or 'person' in c.lower()]
    possible_company = [c for c in df.columns if 'company' in c.lower() or 'issuer' in c.lower() or 'entity' in c.lower()]
    insider_col = possible_insider[0] if possible_insider else df.columns[0]
    company_col = possible_company[0] if possible_company else df.columns[1]
    edges = [(str(row.get(insider_col)), str(row.get(company_col))) for _, row in df.iterrows() if row.get(insider_col) and row.get(company_col)]
    edges_df = pd.DataFrame(edges, columns=['insider','company']).drop_duplicates()
    if save:
        edges_df.to_csv(os.path.join(GRAPH_DIR, 'insider_company_edges.csv'), index=False)
    print(f'✅ Graph edges saved: {edges_df.shape[0]} relations.')
    return edges_df

# -----------------------------
# 7. Main Runner
# -----------------------------
if __name__ == '__main__':
    print('=== START: Data collection run (enhanced) ===')
    tickers = ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS']
    fetch_and_store_market_data(tickers, start='2020-01-01')
    
    try:
        df_sebi = fetch_sebi_disclosures_selenium()
    except Exception as e:
        print('SEBI fetch failed:', e)
    
    try:
        df_ann = fetch_nse_announcements()
    except Exception as e:
        print('NSE fetch failed:', e)

    if 'df_sebi' in locals() and df_sebi is not None:
        build_graph_from_sebi(df_sebi)

    print('=== DONE: All datasets collected ===')

=== START: Data collection run (example) ===
Fetching market data for ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS']


C:\Users\siddh\AppData\Local\Temp\ipykernel_6912\752245282.py:84: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(t, start=start, end=end, interval=interval, progress=False, threads=False)
C:\Users\siddh\AppData\Local\Temp\ipykernel_6912\752245282.py:84: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(t, start=start, end=end, interval=interval, progress=False, threads=False)
C:\Users\siddh\AppData\Local\Temp\ipykernel_6912\752245282.py:84: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(t, start=start, end=end, interval=interval, progress=False, threads=False)
C:\Users\siddh\AppData\Local\Temp\ipykernel_6912\752245282.py:84: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(t, start=start, end=end, interval=interval, progress=False, threads=False)


No table rows found — page may be JS heavy. Consider Selenium or manual download.
NSE announcement endpoint returned status 401
=== DONE ===
Pipeline file created. See data/novelty_notes.txt for publication/patent ideas.
